# Maia Corpus v9 — Acquisition Engine v0.5
**Piloto robusto: Google Drive, fontes independentes, autenticação HF opcional, timeout, progresso e quarentena.**

O OpenWebMath v0.4 já coletou 100 documentos em quarentena. Por padrão não o repetimos. O The Stack Smol requer permissão no Hugging Face. O Gutenberg é consultado via Gutendex, mas todos os livros ficam em quarentena para revisão territorial. **Nenhum documento é aprovado automaticamente para treinamento.**

## 1. Drive persistente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, sys, subprocess, time, signal, datetime
ROOT=Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_4_0/maia-corpus-v9')
BASE=ROOT/'pilot'
for x in ['quarantine','accepted','reports','logs','workers','configs']:
    (BASE/x).mkdir(parents=True,exist_ok=True)
print(ROOT)

## 2. Fontes e limites — ajuste antes de executar

In [ ]:
CFG={
 'max_docs':30,'max_chars':100000,'network_timeout':25,
 'sources':[
  {'id':'openwebmath','enabled':False,'kind':'hf','domain':'math','dataset':'open-web-math/open-web-math','split':'train','text_fields':['text','content']},
  {'id':'the_stack_smol','enabled':True,'kind':'hf','domain':'code','dataset':'bigcode/the-stack-smol','split':'train','text_fields':['content','text','code']},
  {'id':'gutenberg','enabled':True,'kind':'books','domain':'books','languages':['pt','en','es'],'per_language':3},
  {'id':'alternative_code','enabled':False,'kind':'hf','domain':'code','dataset':'REPLACE_AFTER_LICENSE_REVIEW','split':'train','text_fields':['content','text','code']}
 ]}
CONFIG=BASE/'configs'/'v05.json'
CONFIG.write_text(json.dumps(CFG,ensure_ascii=False,indent=2),encoding='utf-8')
print(CONFIG)

## 3. HF_TOKEN opcional
No Colab, abra **Secrets** (ícone de chave), adicione `HF_TOKEN` e habilite o acesso pelo notebook. Antes, aceite os termos de acesso na página https://huggingface.co/datasets/bigcode/the-stack-smol . O token não é gravado no Drive nem exibido.

In [ ]:
HF_TOKEN=None
try:
 from google.colab import userdata
 HF_TOKEN=userdata.get('HF_TOKEN')
except Exception: pass
print('Token configurado:',bool(HF_TOKEN))

## 4. Ambiente isolado com virtualenv (sem modificar requests do Colab)

In [ ]:
VENV=Path('/content/maia-v9-v05-env')
PY=VENV/'bin/python'
if not PY.exists():
 try:
  subprocess.run([sys.executable,'-m','virtualenv','--version'],check=True,capture_output=True,timeout=20)
 except (subprocess.CalledProcessError,subprocess.TimeoutExpired):
  subprocess.run([sys.executable,'-m','pip','install','-q','virtualenv'],check=True,timeout=180)
 subprocess.run([sys.executable,'-m','virtualenv',str(VENV)],check=True,timeout=120)
subprocess.run([str(PY),'-m','pip','install','-q','datasets>=3,<5','huggingface_hub>=0.30,<2','requests>=2.32,<3'],check=True,timeout=600)
subprocess.run([str(PY),'-c','import datasets,requests; print("worker:",datasets.__version__,requests.__version__)'],check=True,timeout=30)
subprocess.run([sys.executable,'-c','import requests; print("colab:",requests.__version__)'],check=True,timeout=30)

## 5. Worker: grava documentos incrementalmente no Drive
Deduplicação SHA-256 contra os arquivos **v0.5** já gravados; não contra o corpus v8/v0.4. Cada documento é gravado com flush, para sobreviver a interrupções. Streaming pode reiniciar do começo em uma nova execução. Cada fonte gera seu próprio relatório.

In [ ]:
WORKER='\nimport argparse,datetime,hashlib,json,os,urllib.parse,urllib.request\nfrom pathlib import Path\ndef choose(row,keys):\n for k in keys:\n  v=row.get(k)\n  if isinstance(v,str) and v.strip(): return v.strip()\n return None\ndef gutenberg(src,cfg):\n for lang in src[\'languages\']:\n  url=\'https://gutendex.com/books/?\'+urllib.parse.urlencode({\'languages\':lang,\'mime_type\':\'text/plain\'})\n  print(\'QUERY\',url,flush=True)\n  req=urllib.request.Request(url,headers={\'User-Agent\':\'MaiaCorpusV9/0.5\'})\n  with urllib.request.urlopen(req,timeout=cfg[\'network_timeout\']) as r:\n   books=json.loads(r.read(3000000).decode())[\'results\']\n  for book in books[:src[\'per_language\']]:\n   fmt=book.get(\'formats\',{})\n   links=[v for k,v in fmt.items() if k.startswith(\'text/plain\') and isinstance(v,str) and v.startswith(\'https://\')]\n   if not links: continue\n   url=links[0]; host=urllib.parse.urlparse(url).hostname\n   if host not in (\'gutenberg.org\',\'www.gutenberg.org\'): continue\n   try:\n    req=urllib.request.Request(url,headers={\'User-Agent\':\'MaiaCorpusV9/0.5\'})\n    with urllib.request.urlopen(req,timeout=cfg[\'network_timeout\']) as r: raw=r.read(2000001)\n    if len(raw)>2000000: continue\n    yield {\'text\':raw.decode(\'utf-8\',errors=\'replace\'),\'url\':url,\'title\':book.get(\'title\'),\n           \'book_id\':book.get(\'id\'),\'languages\':book.get(\'languages\'),\'copyright\':book.get(\'copyright\')}\n   except Exception as exc: print(\'BOOK ERROR\',book.get(\'id\'),str(exc),flush=True)\ndef main():\n ap=argparse.ArgumentParser()\n for name in (\'root\',\'config\',\'source\'): ap.add_argument(\'--\'+name,required=True)\n a=ap.parse_args(); base=Path(a.root)/\'pilot\'; cfg=json.loads(Path(a.config).read_text())\n src=next(x for x in cfg[\'sources\'] if x[\'id\']==a.source)\n stamp=datetime.datetime.now(datetime.timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')\n known=set()\n for folder in (\'accepted\',\'quarantine\'):\n  for path in (base/folder).glob(\'v05_*.jsonl\'):\n   for line in path.open(encoding=\'utf-8\'):\n    try: known.add(json.loads(line)[\'sha256\'])\n    except (ValueError,KeyError): pass\n report={\'source\':src[\'id\'],\'examined\':0,\'written\':0,\'duplicates\':0,\'missing_text\':0,\'error\':None}\n target=base/\'quarantine\'/f"v05_{src[\'id\']}_{stamp}.jsonl"\n try:\n  if src[\'kind\']==\'hf\':\n   from datasets import load_dataset\n   print(\'HF OPEN\',src[\'dataset\'],flush=True)\n   rows=iter(load_dataset(src[\'dataset\'],split=src.get(\'split\',\'train\'),streaming=True,trust_remote_code=False))\n   print(\'HF OPENED\',flush=True)\n  else: rows=gutenberg(src,cfg)\n  with target.open(\'a\',encoding=\'utf-8\') as f:\n   for row in rows:\n    if report[\'examined\']>=cfg[\'max_docs\']: break\n    report[\'examined\']+=1\n    txt=choose(row,src.get(\'text_fields\',[\'text\'])) if src[\'kind\']==\'hf\' else row.get(\'text\')\n    if not txt: report[\'missing_text\']+=1; continue\n    txt=txt[:cfg[\'max_chars\']]; sha=hashlib.sha256(txt.encode()).hexdigest()\n    if sha in known: report[\'duplicates\']+=1; continue\n    known.add(sha)\n    lic=choose(row,[\'license\',\'license_id\',\'spdx_license\'])\n    item={\'source\':src[\'id\'],\'domain\':src[\'domain\'],\'sha256\':sha,\'license_metadata\':lic,\n          \'review_reason\':\'manual_license_and_provenance_review\',\n          \'source_url\':row.get(\'url\') or row.get(\'source_url\'),\'title\':row.get(\'title\'),\n          \'languages\':row.get(\'languages\'),\'text\':txt}\n    f.write(json.dumps(item,ensure_ascii=False)+\'\\n\'); f.flush(); os.fsync(f.fileno())\n    report[\'written\']+=1\n    if report[\'examined\']%5==0: print(\'PROGRESS\',json.dumps(report),flush=True)\n except Exception as exc:\n  report[\'error\']=type(exc).__name__+\': \'+str(exc)\n  print(\'ERROR\',report[\'error\'],flush=True)\n finally:\n  rp=base/\'reports\'/f"v05_{src[\'id\']}_{stamp}.json"\n  rp.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding=\'utf-8\')\n  print(\'REPORT_PATH\',rp,flush=True)\n  print(\'FINAL\',json.dumps(report,ensure_ascii=False),flush=True)\nif __name__==\'__main__\': main()\n'
WORKER_PATH=BASE/'workers'/'maia_v9_worker_v05.py'
WORKER_PATH.write_text(WORKER,encoding='utf-8')
print(WORKER_PATH)

## 6. Execução por fonte, com logs em tempo real e timeout
Timeout de 180 segundos **por fonte**. A interrupção do usuário encerra o grupo do worker. Se uma fonte falhar, a próxima ainda é tentada. Logs e dados são persistentes no Drive.

In [ ]:
SOURCE_TIMEOUT=180
def run_source(src):
 sid=src['id']
 logpath=BASE/'logs'/f"v05_{sid}_{datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.log"
 env=os.environ.copy()
 if HF_TOKEN: env['HF_TOKEN']=HF_TOKEN
 cmd=[str(PY),'-u',str(WORKER_PATH),'--root',str(ROOT),'--config',str(CONFIG),'--source',sid]
 print('INICIANDO',sid,flush=True)
 with logpath.open('w',encoding='utf-8') as log:
  proc=subprocess.Popen(cmd,stdout=log,stderr=subprocess.STDOUT,env=env,start_new_session=True)
  start=time.monotonic(); pos=0; timed_out=False
  try:
   while proc.poll() is None:
    with logpath.open(encoding='utf-8',errors='replace') as r:
     r.seek(pos); chunk=r.read(); pos=r.tell()
    if chunk: print(chunk,end='',flush=True)
    if time.monotonic()-start>SOURCE_TIMEOUT:
     timed_out=True
     print('TIMEOUT:',sid,flush=True)
     os.killpg(proc.pid,signal.SIGTERM)
     try: proc.wait(timeout=5)
     except subprocess.TimeoutExpired:
      os.killpg(proc.pid,signal.SIGKILL); proc.wait()
     break
    time.sleep(1)
  except KeyboardInterrupt:
   if proc.poll() is None:
    os.killpg(proc.pid,signal.SIGTERM)
    try: proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
     os.killpg(proc.pid,signal.SIGKILL); proc.wait()
   raise
  with logpath.open(encoding='utf-8',errors='replace') as r:
   r.seek(pos); print(r.read(),end='',flush=True)
 print('FINALIZADO',sid,'exit=',proc.returncode,'timeout=',timed_out,'log=',logpath,flush=True)
 return {'source':sid,'exit':proc.returncode,'timeout':timed_out,'log':str(logpath)}
results=[]
for source in CFG['sources']:
 if source['enabled']: results.append(run_source(source))
print(json.dumps(results,ensure_ascii=False,indent=2))

## 7. Auditoria dos resultados

In [ ]:
for path in sorted((BASE/'reports').glob('v05_*.json'))[-15:]:
 data=json.loads(path.read_text(encoding='utf-8'))
 print(path.name,'examined=',data['examined'],'written=',data['written'],
       'duplicates=',data['duplicates'],'error=',data['error'])
print('\nAtenção: todos os novos documentos permanecem em quarentena até revisão.')

## 8. Próximas validações
- Confirmar acesso autorizado ao The Stack Smol ou selecionar fonte alternativa auditada.
- Revisar licenças **por documento** (inclusive OpenWebMath) e direitos territoriais dos livros.
- Deduplicar contra o corpus v8 e v0.4; verificar qualidade e separação treino/validação.
- Retokenizar com MaiaTokenizer-65k e calcular volumes reais de tokens.

**Este notebook é um piloto, não um coletor de produção de 10 bilhões de tokens.**